# Colab Training Launcher

This notebook launches the package training scripts. It does not contain model or training logic.

Expected Drive dataset path:

```text
/content/drive/MyDrive/VOC_Dataset/voc2012_processed/
```


In [ ]:
import yaml
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/lovisticsdev/multilabel_image_classifier_web.git'

%cd /content
!rm -rf app
!git clone $REPO_URL app
%cd /content/app
!git log -1 --oneline


In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!pip install -e '.[dev,train]'


In [ ]:
from multilabel_classifier.data.datamodule import create_dataloaders
print('datamodule import OK:', create_dataloaders)


In [ ]:
config_path = Path('configs/voc20.yaml')

dataset_root = Path('/content/drive/MyDrive/VOC_Dataset/voc2012_processed')

paths_to_check = {
    'train_images': dataset_root / 'train' / 'images',
    'val_images': dataset_root / 'val' / 'images',
    'train_annotations': dataset_root / 'train_annotations.csv',
    'val_annotations': dataset_root / 'val_annotations.csv',
}

for name, path in paths_to_check.items():
    if not path.exists():
        raise FileNotFoundError(f'{name} not found: {path}')

cfg = yaml.safe_load(config_path.read_text())

cfg['dataset']['train_images'] = str(paths_to_check['train_images'])
cfg['dataset']['val_images'] = str(paths_to_check['val_images'])
cfg['dataset']['train_annotations'] = str(paths_to_check['train_annotations'])
cfg['dataset']['val_annotations'] = str(paths_to_check['val_annotations'])

cfg.setdefault('outputs', {})
cfg['outputs']['checkpoint_dir'] = '/content/drive/MyDrive/VOC_Dataset/checkpoints'
cfg['outputs']['export_dir'] = '/content/drive/MyDrive/VOC_Dataset/exported_model'
cfg['outputs']['report_dir'] = '/content/drive/MyDrive/VOC_Dataset/model_outputs'

config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(config_path.read_text())


In [ ]:
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Training will run on CPU and may be slow.')


In [ ]:
!python scripts/inspect_dataset.py --config configs/voc20.yaml


In [ ]:
!python scripts/train.py --config configs/voc20.yaml


In [ ]:
!python scripts/evaluate.py --config configs/voc20.yaml --artifact-dir /content/drive/MyDrive/VOC_Dataset/exported_model
